# Checking each cleaning function (Peter-work)

**Vision:** a junior equity research analyst decides which of seven mega-cap technology stocks (AAPL, MSFT, GOOGL, AMZN, NVDA, META, TSLA) to prioritize for this week's deep-dive research.
**Question:** how does company news relate to subsequent price changes and trading volume?

`clean_peter.py` holds the functions; this notebook calls them one at a time and checks the result. Rules and decisions (D1 to D10) are in `plan.md`.

Each section: **what the function should do**, a run on the real data, then `assert` checks. If an assert fails the cell stops with an error, so running the whole notebook without an error means every check passed.

Run from the project folder with the raw files in `data/raw/`: Restart the kernel, then Run All.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import clean_peter as cp

pd.set_option("display.max_colwidth", 80)
prices_all, news_all, filings_all = cp.load_raw()
pd.DataFrame({"rows": [len(prices_all), len(news_all), len(filings_all)],
              "cols": [prices_all.shape[1], news_all.shape[1], filings_all.shape[1]],
              "symbols": [d["symbol"].nunique() for d in (prices_all, news_all, filings_all)]},
             index=["prices", "news", "filings"])

,rows,cols,symbols
prices,2656,9,8
news,3888,4,8
filings,73,6,8


## 1. `apply_scope` (D7)
Keeps the seven stocks in the vision and returns the excluded rows separately, so the removal can be counted instead of disappearing.

In [2]:
prices_raw, prices_out = cp.apply_scope(prices_all)
news_raw, news_out = cp.apply_scope(news_all)
filings_raw, filings_out = cp.apply_scope(filings_all)

for name, kept, out, full in [("prices", prices_raw, prices_out, prices_all),
                              ("news", news_raw, news_out, news_all),
                              ("filings", filings_raw, filings_out, filings_all)]:
    assert len(kept) + len(out) == len(full)
    assert set(out["symbol"]) == {"ADBE"} and set(kept["symbol"]) == set(cp.SCOPE_SYMBOLS)
    print(f"{name:8} {len(full)} = {len(kept)} kept + {len(out)} ADBE excluded")

prices   2656 = 2324 kept + 332 ADBE excluded
news     3888 = 3402 kept + 486 ADBE excluded
filings  73 = 63 kept + 10 ADBE excluded


## 2. `clean_common`
Shared rules: `id` to int64, `symbol` stripped and uppercased as category, `date` to datetime64, sorted by symbol and date. Must never drop rows.

First a tiny hand-made table with messy values, so we can see exactly what changes.

In [3]:
toy = pd.DataFrame({
    "id": [3, 1, 2],
    "symbol": [" msft", "AAPL ", "aapl"],
    "date": ["2025-01-03", "2025-01-02", "2025-01-01"],
})
toy_clean = cp.clean_common(toy)
toy_clean

,id,symbol,date
0,2,AAPL,2025-01-01
1,1,AAPL,2025-01-02
2,3,MSFT,2025-01-03


In [4]:
assert list(toy_clean["symbol"]) == ["AAPL", "AAPL", "MSFT"]
assert list(toy_clean["date"].dt.day) == [1, 2, 3]          # sorted by symbol, then date
assert str(toy_clean["date"].dtype).startswith("datetime64")
assert len(toy_clean) == len(toy)
assert toy["symbol"].iloc[0] == " msft"                     # original table untouched (.copy())
print("clean_common: all checks passed")

clean_common: all checks passed


## 3. `latest_date_mentioned`
Finds the latest explicit date written in a text, in `2025-02-25` or `February 25, 2025` form. Used for the look-ahead flag (refinement 2).

In [5]:
cases = {
    "Closed at 239.58 on 2024-12-02.": "2024-12-02",
    "On February 25, Adobe launched ... reported Feb 25, 2025 and 2025-02-14.": "2025-02-25",
    "No explicit date here.": None,
    "Avoids facts dated after 2025-10-14.": "2025-10-14",
}
for text, expected in cases.items():
    got = cp.latest_date_mentioned(text)
    print(f"{str(got.date()) if pd.notna(got) else None!s:12}  <- {text}")
    assert (pd.isna(got) and expected is None) or str(got.date()) == expected
print("latest_date_mentioned: all checks passed")

2024-12-02    <- Closed at 239.58 on 2024-12-02.
2025-02-25    <- On February 25, Adobe launched ... reported Feb 25, 2025 and 2025-02-14.
None          <- No explicit date here.
2025-10-14    <- Avoids facts dated after 2025-10-14.
latest_date_mentioned: all checks passed


The last case is a disclaimer that only returns the row's own date, so it is **not** flagged: the flag needs a date later than the row date. That is the false positive refinement 2 removed.

## 4. `clean_prices`
Adds `daily_return` from `adj_close` (D6), `flag_large_move` for moves above 12% (D5), and `volume_ratio_20d`: today's volume divided by the average of the **previous** 20 sessions (D10), so 1.0 is normal and 3.0 is three times normal.

In [6]:
prices = cp.clean_prices(prices_raw)
print(prices.shape)
prices.head(3)

(2324, 12)


,id,symbol,date,open,high,low,close,adj_close,volume,daily_return,flag_large_move,volume_ratio_20d
0,1,AAPL,2024-12-02,237.270004,240.789993,237.160004,239.589996,238.288300,48137100,NaN,False,NaN
1,2,AAPL,2024-12-03,239.809998,242.759995,238.899994,242.649994,241.331665,38861000,0.012772,False,NaN
2,3,AAPL,2024-12-04,242.869995,244.110001,241.250000,243.009995,241.689728,44383900,0.001484,False,NaN


In [7]:
assert len(prices) == len(prices_raw)
assert prices["daily_return"].isna().sum() == prices["symbol"].nunique()          # first session only
assert prices["volume_ratio_20d"].isna().sum() == cp.VOLUME_WINDOW * prices["symbol"].nunique()
assert cp.ohlc_violations(prices) == 0

# recompute one return and one volume ratio by hand
aapl = prices[prices["symbol"] == "AAPL"].reset_index(drop=True)
assert abs(aapl["adj_close"][1] / aapl["adj_close"][0] - 1 - aapl["daily_return"][1]) < 1e-12
assert abs(aapl["volume"][20] / aapl["volume"][0:20].mean() - aapl["volume_ratio_20d"][20]) < 1e-12
print("clean_prices: all checks passed")

clean_prices: all checks passed


In [8]:
prices.loc[prices["flag_large_move"], ["symbol", "date", "close", "daily_return", "volume_ratio_20d"]]

,symbol,date,close,daily_return,volume_ratio_20d
87,AAPL,2025-04-09,198.850006,0.153288,2.814449
1083,META,2025-04-09,585.770020,0.147556,1.925233
1696,NVDA,2025-01-27,118.419998,-0.169682,4.076792
1747,NVDA,2025-04-09,114.330002,0.187227,2.009133
2057,TSLA,2025-03-10,222.149994,-0.154262,2.062835
2079,TSLA,2025-04-09,272.200012,0.226900,1.540101
2118,TSLA,2025-06-05,284.700012,-0.142599,2.751133


All of these match known market events (2025-04-09 tariff pause rally, NVDA DeepSeek drop on 2025-01-27), so they stay with a flag. The biggest volume spikes:

In [9]:
prices.nlargest(3, "volume_ratio_20d")[["symbol", "date", "daily_return", "volume_ratio_20d"]]

,symbol,date,daily_return,volume_ratio_20d
1224,META,2025-10-30,-0.113348,7.190962
1617,MSFT,2026-01-29,-0.099931,4.897704
1696,NVDA,2025-01-27,-0.169682,4.076792


Which of the sessions above 4x normal volume come right after a filing? This is the kind of news-to-volume link the project question asks about.

In [10]:
spikes = prices.loc[prices["volume_ratio_20d"] > 4, ["symbol", "date", "daily_return", "volume_ratio_20d"]]
filings_tmp = cp.clean_filings(filings_raw, cp.trading_calendar(prices))
spikes["session_after_filing"] = [((filings_tmp["symbol"] == s) & (filings_tmp["reaction_date"] == d)).any()
                                  for s, d in zip(spikes["symbol"], spikes["date"])]
spikes

,symbol,date,daily_return,volume_ratio_20d,session_after_filing
627,AMZN,2026-02-06,-0.055548,4.022438,True
1224,META,2025-10-30,-0.113348,7.190962,True
1285,META,2026-01-29,0.104048,4.014540,True
1617,MSFT,2026-01-29,-0.099931,4.897704,True
1696,NVDA,2025-01-27,-0.169682,4.076792,False


## 5. `trading_calendar` and `map_to_session` (D9)
`map_to_session` answers "which trading session does this date belong to?" in two ways: the same or next session (`next_trading_date`), and the first session strictly **after** the date (`reaction_date`, the subsequent session in the business question). Dates past the last session, or well before the first, get NaT because their real session is not in the data.

In [11]:
days = cp.trading_calendar(prices)
assert days.is_monotonic_increasing and days.is_unique and (days.dt.dayofweek < 5).all()

test_dates = ["2025-02-14", "2025-02-15", "2025-02-17", "2025-02-18", "2026-03-31", "2024-12-01", "2024-11-20"]
demo = pd.DataFrame({
    "date": pd.to_datetime(test_dates),
    "same_or_next": cp.map_to_session(test_dates, days, strictly_after=False),
    "strictly_after": cp.map_to_session(test_dates, days, strictly_after=True),
})
demo["weekday"] = demo["date"].dt.day_name()
demo

,date,same_or_next,strictly_after,weekday
0,2025-02-14,2025-02-14,2025-02-18,Friday
1,2025-02-15,2025-02-18,2025-02-18,Saturday
2,2025-02-17,2025-02-18,2025-02-18,Monday
3,2025-02-18,2025-02-18,2025-02-19,Tuesday
4,2026-03-31,2026-03-31,NaT,Tuesday
5,2024-12-01,2024-12-02,2024-12-02,Sunday
6,2024-11-20,NaT,NaT,Wednesday


In [12]:
expected_after = ["2025-02-18", "2025-02-18", "2025-02-18", "2025-02-19", None, "2024-12-02", None]
for got, exp in zip(demo["strictly_after"], expected_after):
    assert (pd.isna(got) and exp is None) or str(got.date()) == exp
print("map_to_session: all checks passed (Feb 17 2025 was Presidents Day; 2026-03-31 is the last session)")

map_to_session: all checks passed (Feb 17 2025 was Presidents Day; 2026-03-31 is the last session)


## 6. `clean_news`
Wording is never edited (D3). Adds `text_len`, `flag_placeholder` (D1), `latest_date_mentioned` and `flag_lookahead` (D3), `is_trading_day`, `next_trading_date` (D2) and `reaction_date` (D9).

In [13]:
news = cp.clean_news(news_raw, days)
print(news.shape)
news.drop(columns="highlights").head(3)

(3402, 11)


,id,symbol,date,text_len,flag_placeholder,latest_date_mentioned,flag_lookahead,is_trading_day,next_trading_date,reaction_date
0,1,AAPL,2024-12-01,1586,False,2024-12-01,False,False,2024-12-02,2024-12-02
1,2,AAPL,2024-12-02,2077,False,2024-12-02,False,True,2024-12-02,2024-12-03
2,3,AAPL,2024-12-03,1081,False,NaT,False,True,2024-12-03,2024-12-04


In [14]:
assert len(news) == len(news_raw)
raw_text = news_raw.sort_values(["symbol", "date"])["highlights"].str.strip().reset_index(drop=True)
assert raw_text.equals(news["highlights"])                                 # wording unchanged
assert news["next_trading_date"].notna().all()
assert (news["reaction_date"].dropna() > news.loc[news["reaction_date"].notna(), "date"]).all()
assert (news.loc[news["reaction_date"].isna(), "date"] == days.max()).all()  # only the last day lacks one
print("clean_news: all checks passed;", int(news["flag_lookahead"].sum()), "look-ahead rows in scope")

clean_news: all checks passed; 0 look-ahead rows in scope


**Placeholder rows.** Each should be an empty or broken summary, not real news.

In [15]:
news.loc[news["flag_placeholder"], ["symbol", "date", "text_len", "highlights"]]

,symbol,date,text_len,highlights
518,AMZN,2025-01-02,153,"This item conveys a clear negative signal for that specific date, indicating..."
982,GOOGL,2024-12-11,352,I can't provide any historical news or updates regarding GOOGL Google Alphab...
1000,GOOGL,2024-12-29,397,"There are no substantive GOOGL news items, events, filings, earnings, or mar..."
1002,GOOGL,2024-12-31,138,"Consequently, there are no substantive GOOGL events, themes, or market senti..."
1471,META,2024-12-14,222,"The sole\n\nAs a result, no themes, developments, or market sentiment relate..."
1475,META,2024-12-18,288,"Because no META content is present in the supplied material for that date, t..."
1963,MSFT,2024-12-20,374,"Because no substantive MSFT events, earnings, product launches, analyst note..."


## 7. `clean_filings`
Decodes HTML entities (`&#8217;` to a right quote, `&#160;` to a space) and strips whitespace (D8). Adds `mda_len`, `risk_len`, `flag_risk_repeat`, `flag_risk_reference_only` (D4), `reaction_date` and `in_price_window` (D9).

In [16]:
filings = cp.clean_filings(filings_raw, days)
print(filings.shape)
filings.drop(columns=["mda_content", "risk_content"]).head(3)

(63, 12)


,id,symbol,date,document_type,mda_len,risk_len,flag_risk_repeat,flag_risk_reference_only,reaction_date,in_price_window
0,1,AAPL,2024-02-01,10-Q,15470,4451,False,False,NaT,False
1,2,AAPL,2024-05-02,10-Q,17027,4651,False,False,NaT,False
2,3,AAPL,2024-08-01,10-Q,17616,940,False,True,NaT,False


In [17]:
assert len(filings) == len(filings_raw)
assert set(filings["document_type"].unique()) <= {"10-K", "10-Q"}
for col in ["mda_content", "risk_content"]:
    assert not filings[col].str.contains(cp.HTML_ENTITY, regex=True).any()
    assert (filings[col] == filings[col].str.strip()).all()
assert not filings.groupby("symbol", observed=True)["flag_risk_repeat"].first().any()  # first filing can't repeat
assert (filings.loc[~filings["in_price_window"], "date"] < days.min()).all()
print("before:", repr(filings_raw.sort_values(["symbol", "date"])["mda_content"].iloc[0][:60]))
print("after: ", repr(filings["mda_content"].iloc[0][:60]))
print("clean_filings: all checks passed")

before: ' Item 2. Management&#8217;s Discussion and Analysis of Finan'
after:  'Item 2. Management’s Discussion and Analysis of Financial Co'
clean_filings: all checks passed


In [18]:
filings.loc[filings["flag_risk_repeat"] | filings["flag_risk_reference_only"],
            ["symbol", "date", "document_type", "risk_len", "flag_risk_repeat", "flag_risk_reference_only", "in_price_window"]]

,symbol,date,document_type,risk_len,flag_risk_repeat,flag_risk_reference_only,in_price_window
2,AAPL,2024-08-01,10-Q,940,False,True,False
8,AAPL,2026-01-30,10-Q,412,False,True,True
12,AMZN,2024-10-31,10-Q,59365,True,False,False
15,AMZN,2025-07-31,10-Q,60089,True,False,True
16,AMZN,2025-10-30,10-Q,60089,True,False,True
23,GOOGL,2025-04-24,10-Q,591,False,True,True
24,GOOGL,2025-07-23,10-Q,591,True,True,True
25,GOOGL,2025-10-29,10-Q,591,True,True,True
55,TSLA,2024-04-23,10-Q,329,False,True,False
56,TSLA,2024-07-23,10-Q,329,True,True,False


## 8. Before vs after
`compare_profiles` and `reconcile_rows` account for every row; `flag_summary` counts every flag by stock.

In [19]:
raw = {"prices": prices_all, "news": news_all, "filings": filings_all}
excluded = {"prices": prices_out, "news": news_out, "filings": filings_out}
clean = {"prices": prices, "news": news, "filings": filings}
rec = cp.reconcile_rows(raw, excluded, clean)
assert (rec["other_rows_removed"] == 0).all()
rec

,start_rows,excluded_out_of_scope,other_rows_removed,final_rows
table,,,,
prices,2656,332,0,2324
news,3888,486,0,3402
filings,73,10,0,63


In [20]:
cp.compare_profiles(raw, clean)

,rows_before,missing_before,dup_id_before,dup_symbol_date_before,symbols_before,rows_after,missing_after,dup_id_after,dup_symbol_date_after,symbols_after
table,,,,,,,,,,
prices,2656,0,0,0,8,2324,147,0,0,7
news,3888,0,0,0,8,3402,409,0,0,7
filings,73,0,0,0,8,63,28,0,0,7


In [21]:
cp.flag_summary(prices, news, filings)

,flag_large_move,flag_placeholder,flag_lookahead,non_trading_news,flag_risk_repeat,flag_risk_reference_only,filings_before_price_window
symbol,,,,,,,
AAPL,1,0,0,154,0,2,4
AMZN,0,1,0,154,3,0,4
GOOGL,0,3,0,154,2,3,4
META,1,2,0,154,0,0,4
MSFT,0,1,0,154,0,0,4
NVDA,2,0,0,154,0,0,4
TSLA,3,0,0,154,2,4,4
TOTAL,7,7,0,1078,7,9,28


## 9. `join_news_to_prices` (checklist item 9)
Joins each news row to the **subsequent** session: news (symbol, reaction_date) to prices (symbol, date). Expected: many news rows to one price row, no row multiplication, and only last-day news unmatched. `validate="many_to_one"` makes pandas raise an error if a price key were duplicated.

In [22]:
joined, join_stats = cp.join_news_to_prices(news, prices)
assert join_stats["rows_after_join"] == join_stats["news_rows_before"]     # no multiplication
assert join_stats["unmatched_with_a_session"] == 0
assert join_stats["news_without_next_session"] == prices["symbol"].nunique()
join_stats

{'news_rows_before': 3402,
 'rows_after_join': 3402,
 'news_without_next_session': 7,
 'unmatched_with_a_session': 0,
 'max_news_rows_per_price_row': 4}

## 10. `metric_support` (checklist item 11)
How many records support each planned metric, and why the rest are excluded.

In [23]:
support = cp.metric_support(prices, news, filings)
assert (support["missing_group_keys"] == 0).all()
assert (support["records_used"] + support["records_excluded"] == support["records_total"]).all()
support

,metric,table,records_total,records_used,records_excluded,excluded_because,missing_group_keys
0,daily return and volatility per stock,prices,2324,2317,7,first session of each stock has no prior price,0
1,volume ratio vs previous 20 sessions,prices,2324,2184,140,first 20 sessions of each stock have no full baseline,0
2,next-session return after news,news,3402,3381,21,placeholder or look-ahead news; no next session in the data (last day); or t...,0
3,next-session volume ratio after news,news,3402,3190,212,"placeholder or look-ahead news, no next session, or next session inside the ...",0
4,price and volume reaction after a filing,filings,63,35,28,filed before the price data starts (2024-12-02),0


## 11. `record_checks` (checklist item 12)
Six difficult records traced from the original data to the cleaned data. Expected results are written by hand (check 2 recomputes return and volume ratio from the raw table); actual results are read from the cleaned data.

In [24]:
checks = cp.record_checks(raw, clean)
assert checks["match"].all()
checks[["check", "table", "symbol", "date", "case", "expected", "actual", "match"]]

,check,table,symbol,date,case,expected,actual,match
0,1,prices,AAPL,2024-12-02,first session in the data,kept; close 239.59; daily_return NaN; volume_ratio_20d NaN,kept; close 239.59; daily_return NaN; volume_ratio_20d NaN,True
1,2,prices,NVDA,2025-01-27,largest one-day drop (DeepSeek news),kept; daily_return -0.1697; volume_ratio_20d 4.08; flag_large_move True,kept; daily_return -0.1697; volume_ratio_20d 4.08; flag_large_move True,True
2,3,news,GOOGL,2024-12-29,placeholder summary on a Sunday,kept; text unchanged; flag_placeholder True; next_trading_date 2024-12-30; r...,kept; text unchanged; flag_placeholder True; next_trading_date 2024-12-30; r...,True
3,4,news,AAPL,2025-02-14,Friday before a weekend and a market holiday,kept; is_trading_day True; next_trading_date 2025-02-14; reaction_date 2025-...,kept; is_trading_day True; next_trading_date 2025-02-14; reaction_date 2025-...,True
4,5,filings,TSLA,2024-07-23,"10-Q before the price window, risk section only points to the 10-K",kept; risk_len 329; same as previous filing; both risk flags True; in_price_...,kept; risk_len 329; same as previous filing; both risk flags True; in_price_...,True
5,6,news,ADBE,2025-02-15,company outside the vision scope,present in raw; absent from cleaned; counted in excluded_out_of_scope,present in raw; absent from cleaned; counted in excluded_out_of_scope,True


## 12. `make_sample`
Inspection sample of cleaned rows with all columns, at most 50 rows, weighted toward difficult cases. Every row has a `sample_reason`. The checks above always run on the full data, not on this sample.

In [25]:
sample = cp.make_sample(prices, news, filings)
assert sum(len(v) for v in sample.values()) <= 50
pd.concat({k: v["sample_reason"].value_counts() for k, v in sample.items()})

         sample_reason                                                    
prices   large move > 12% (flagged, kept)                                      7
         first session: daily_return and volume ratio NaN by design            7
         ordinary day (control)                                                2
         first session with a full 20-day volume baseline                      1
news     placeholder or fragment (flagged, kept)                               7
         ordinary trading day (control)                                        2
         Friday before weekend and holiday: reaction_date skips to Tuesday     1
         weekday market holiday                                                1
         weekend news                                                          1
         last day: no next session in the data                                 1
filings  risk section repeated or reference-only (flagged, kept)              12
         ordinary filing inside th

## 13. Full pipeline
`run_pipeline` chains everything, saves `data/clean/*.parquet`, the sample in `samples_peter/`, and the reports in `outputs/`. Reading the saved files back should give the same tables checked above.

In [26]:
out = cp.run_pipeline()
for name in ["prices", "news", "filings"]:
    saved = pd.read_parquet(f"data/clean/{name}_clean.parquet")
    assert saved.equals(clean[name]), name
    print(f"{name:8} saved {saved.shape}")
print("pipeline: all checks passed")

prices   saved (2324, 12)
news     saved (3402, 11)
filings  saved (63, 12)
pipeline: all checks passed
